# K-means en 2D: dos distribuciones cercanas

Este notebook presenta un caso más exigente para K-means: dos grupos gaussianos están más cercanos entre sí. La intención es observar que el agrupamiento se vuelve más difícil cuando las distribuciones se solapan.

## 1. Planteamiento

K-means separa los datos según distancia a centroides. Por eso funciona bien cuando los grupos están compactos y separados. Cuando los grupos se acercan, algunos puntos pueden quedar en una zona ambigua.

La frontera de decisión inducida por dos centroides es aproximadamente una línea perpendicular al segmento que une los centroides. Por tanto, los puntos cercanos a esa frontera son los más propensos a cambiar de cluster.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. Generación de datos

Se crean dos distribuciones gaussianas con medias cercanas:

$$
\boldsymbol{\mu}_1 = [2,3], \qquad \boldsymbol{\mu}_2 = [3,4]
$$

A diferencia del primer ejemplo, la distancia entre los centros reales de las distribuciones es menor. Esto incrementa la dificultad del problema.

In [ ]:
mu1 = np.array([2, 3])
sigma1 = np.array([[0.5, 0.3],
                   [0.3, 0.5]])

mu2 = np.array([3, 4])
sigma2 = np.array([[0.5, -0.2],
                   [-0.2, 0.7]])

n_points = 100

data1 = rng.multivariate_normal(mu1, sigma1, n_points)
data2 = rng.multivariate_normal(mu2, sigma2, n_points)

data = np.vstack([data1, data2])

df = pd.DataFrame(data, columns=["dimension_1", "dimension_2"])
df.head()

## 3. Datos sin etiquetas

Aunque internamente sabemos que los datos fueron simulados desde dos distribuciones, el algoritmo solo recibe la matriz de características. No recibe la información de cuál punto pertenece a cuál distribución.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], s=40, alpha=0.8)
plt.title("Datos 2D sin etiquetar: distribuciones cercanas")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.show()

## 4. Entrenamiento de K-means

Se solicita al algoritmo encontrar dos clusters:

$$
K = 2
$$

El modelo alterna entre dos pasos:

1. asignar cada punto al centroide más cercano;
2. recalcular cada centroide como el promedio de los puntos asignados.

In [ ]:
num_clusters = 2

kmeans = KMeans(n_clusters=num_clusters, random_state=RANDOM_STATE, n_init=10)
cluster_idx = kmeans.fit_predict(data)
cluster_centers = kmeans.cluster_centers_

pd.DataFrame(cluster_centers, columns=["centroide_x", "centroide_y"])

## 5. Resultado del agrupamiento

Cuando las distribuciones están cercanas, K-means todavía puede encontrar dos regiones, pero la separación ya no coincide necesariamente con la separación generadora original. La razón es que K-means solo considera distancias geométricas, no conoce el proceso estadístico usado para crear los datos.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=cluster_idx, cmap="tab10", s=40, alpha=0.85)
plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], marker="*", s=300, c="black", edgecolor="white", label="Centroides")
plt.title("K-means con dos distribuciones cercanas")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.legend()
plt.show()

## 6. Actividad propuesta

Incremente el número de clusters modificando `num_clusters`. Por ejemplo, pruebe con:

$$
K = 3, 4, 5
$$

Observe que aumentar $K$ no significa necesariamente mejorar la interpretación del problema. Puede generar subdivisiones artificiales dentro de una misma nube de datos.

In [ ]:
# Experimento sugerido: cambie este valor y vuelva a ejecutar la celda.
num_clusters_experimento = 3

kmeans_exp = KMeans(n_clusters=num_clusters_experimento, random_state=RANDOM_STATE, n_init=10)
cluster_exp = kmeans_exp.fit_predict(data)
centers_exp = kmeans_exp.cluster_centers_

plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=cluster_exp, cmap="tab10", s=40, alpha=0.85)
plt.scatter(centers_exp[:, 0], centers_exp[:, 1], marker="*", s=300, c="black", edgecolor="white", label="Centroides")
plt.title(f"Experimento con K = {num_clusters_experimento}")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.legend()
plt.show()